In [1]:
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
%matplotlib tk

In [2]:
t = sp.Symbol('t')
g = sp.Symbol('g', positive=True)

def make_pendulum(i):
    m = sp.Symbol(f'm{i}', positive=True)
    l = sp.Symbol(f'l{i}', positive=True)
    theta = sp.Function(f'theta{i}')(t)
    return m, l, theta

In [3]:
def get_positions(thetas, lengths):
    x, y = sp.Integer(0), sp.Integer(0)
    positions = []
    
    for i, (theta, l) in enumerate(zip(thetas, lengths)):
        x = x + l * sp.sin(theta)
        y = y - l * sp.cos(theta)
        positions.append((x, y))
    
    return positions

In [4]:
def get_lagrangian(masses, lengths, n):
    thetas = []
    ls = []
    ms = []
    
    for i in range(n):
        m, l, theta = make_pendulum(i + 1)
        thetas.append(theta)
        ls.append(l)
        ms.append(m)
    
    positions = get_positions(thetas, ls)
    
    L = sp.Integer(0)
    
    for i in range(n):
        x, y = positions[i]
        
        # speed
        vx = sp.diff(x, t)
        vy = sp.diff(y, t)
        
        #kinetic energy
        T_i = sp.Rational(1, 2) * ms[i] * (vx**2 + vy**2)
        
        # potencial energy
        V_i = ms[i] * g * y
        
       
         #Lagrangian sum
        L = L + T_i - V_i
    
    return L, thetas, ms, ls

In [5]:
def get_equations_of_motion(L, thetas):
    eqs = []
    
    for theta in thetas:
        theta_dot = sp.diff(theta, t)
        
        # Euler-Lagrange equals zero
        dL_dtheta = sp.diff(L, theta)
        dL_dtheta_dot = sp.diff(L, theta_dot)
        d_dt_dL_dtheta_dot = sp.diff(dL_dtheta_dot, t)
        
        eq = d_dt_dL_dtheta_dot - dL_dtheta
        eqs.append(eq)
    
    return eqs

In [6]:
def solve_for_accelerations(eqs, thetas):
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    theta_ddots = [sp.diff(sp.diff(theta, t), t) for theta in thetas]
    
   #like Gaussian elimination
    solution = sp.solve(eqs, theta_ddots)
    
    return solution, theta_ddots

In [7]:
def build_numerical_system(solution, thetas, masses, lengths, g_val=9.81, n=2):
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    theta_ddots = [sp.diff(sp.diff(theta, t), t) for theta in thetas]
    
    # replace with real values
    subs = {g: g_val}
    for i in range(n):
        subs[masses[i]] = masses[i]
        subs[lengths[i]] = lengths[i]
    
    # all variables
    all_vars = thetas + theta_dots
    
    # fast numeric func
    acc_funcs = []
    for ddot in theta_ddots:
        expr = solution[ddot].subs(subs)
        expr = sp.simplify(expr)
        f = sp.lambdify(all_vars, expr, modules='numpy')
        acc_funcs.append(f)
    
    return acc_funcs

In [8]:
def simulate(acc_funcs, initial_angles, initial_velocities, t_span=(0, 20), n=2):
    y0 = list(initial_angles) + list(initial_velocities)

    def system(t_val, y):
        angles = y[:n]
        velocities = y[n:]
        accelerations = [f(*angles, *velocities) for f in acc_funcs]
        return list(velocities) + list(accelerations)

    result = solve_ivp(
        system,
        t_span,
        y0,
        method='RK45',
        max_step=0.01,
        rtol=1e-8,
        atol=1e-8
    )

    return result

In [9]:
def run_simulation(n, mass_values, length_values, angle_values, velocity_values=None, t_span=(0, 20)):
    if velocity_values is None:
        velocity_values = [0.0] * n

    L, thetas, masses, lengths = get_lagrangian(None, None, n)
    eqs = get_equations_of_motion(L, thetas)
    solution, theta_ddots = solve_for_accelerations(eqs, thetas)

    subs = {g: 9.81}
    for i in range(n):
        subs[masses[i]] = mass_values[i]
        subs[lengths[i]] = length_values[i]

    solution_numeric = {k: v.subs(subs) for k, v in solution.items()}
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    all_vars = thetas + theta_dots

    acc_funcs = []
    for ddot in theta_ddots:
        expr = sp.simplify(solution_numeric[ddot])
        f = sp.lambdify(all_vars, expr, modules='numpy')
        acc_funcs.append(f)

    result = simulate(acc_funcs, angle_values, velocity_values, t_span, n)
    return result, length_values

In [10]:
def animate_pendulum(result, length_values, n, title="N-Махало"):
    t_vals = result.t
    angles = result.y[:n]

    xs = np.zeros((n + 1, len(t_vals)))
    ys = np.zeros((n + 1, len(t_vals)))

    for frame in range(len(t_vals)):
        x, y = 0.0, 0.0
        for i in range(n):
            x = x + length_values[i] * np.sin(angles[i, frame])
            y = y - length_values[i] * np.cos(angles[i, frame])
            xs[i + 1, frame] = x
            ys[i + 1, frame] = y

    fig, ax = plt.subplots(figsize=(6, 6))
    total_length = sum(length_values)
    ax.set_xlim(-total_length * 1.2, total_length * 1.2)
    ax.set_ylim(-total_length * 1.2, total_length * 1.2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

    line, = ax.plot([], [], 'o-', lw=2, markersize=8, color='royalblue')
    trace, = ax.plot([], [], '-', lw=0.5, alpha=0.4, color='tomato')
    trace_x, trace_y = [], []

    def update(frame):
        line.set_data(xs[:, frame], ys[:, frame])
        trace_x.append(xs[-1, frame])
        trace_y.append(ys[-1, frame])
        trace.set_data(trace_x[-300:], trace_y[-300:])
        return line, trace

    ani = FuncAnimation(fig, update, frames=range(0, len(t_vals), 3),
                        interval=20, blit=True)
    plt.tight_layout()
    plt.show()
    return ani

In [11]:
def get_derivatives(t_val, y, n, masses, lengths, g=9.81):
    angles = y[:n]
    omegas = y[n:]

    M = np.zeros((n, n))
    f = np.zeros(n)

    for i in range(n):
        for j in range(n):
            m_sum = sum(masses[k] for k in range(max(i, j), n))
            M[i, j] = m_sum * lengths[i] * lengths[j] * np.cos(angles[i] - angles[j])

        m_sum_i = sum(masses[k] for k in range(i, n))
        f[i] = -sum(
            sum(masses[k] for k in range(max(i, j), n)) *
            lengths[i] * lengths[j] * omegas[j]**2 * np.sin(angles[i] - angles[j])
            for j in range(n)
        ) - m_sum_i * g * lengths[i] * np.sin(angles[i])

    alphas = np.linalg.solve(M, f)
    return np.concatenate([omegas, alphas])


def simulate_fast(n, masses, lengths, angles, omegas=None, t_span=(0, 30)):
    if omegas is None:
        omegas = [0.0] * n

    y0 = np.array(list(angles) + list(omegas))

    result = solve_ivp(
        get_derivatives,
        t_span,
        y0,
        args=(n, masses, lengths),
        method='RK45',
        max_step=0.01,
        rtol=1e-8,
        atol=1e-8
    )
    return result


def lyapunov_fast(n, masses, lengths, angles, t_span=(0, 30)):
    r1 = simulate_fast(n, masses, lengths, angles, t_span=t_span)

    angles_2 = list(angles)
    angles_2[0] += 1e-8
    r2 = simulate_fast(n, masses, lengths, angles_2, t_span=t_span)

    min_len = min(len(r1.t), len(r2.t))
    diff = np.zeros(min_len)
    for i in range(n):
        diff += (r1.y[i, :min_len] - r2.y[i, :min_len])**2
    diff = np.sqrt(diff)
    diff = diff[diff > 0]
    t_vals = r1.t[:len(diff)]
    log_diff = np.log(diff / 1e-8)
    lyap = np.polyfit(t_vals, log_diff, 1)[0]
    return lyap

In [12]:
try:
    N_ANIM = int(input("Колко махала да анимираме? (2-10): "))
    N_ANIM = max(2, N_ANIM)
except ValueError:
    N_ANIM = 2

masses_anim = [1.0] * N_ANIM
lengths_anim = [1.0] * N_ANIM
angles_anim = [np.pi/2] * N_ANIM


result_anim = simulate_fast(N_ANIM, masses_anim, lengths_anim, angles_anim, t_span=(0, 20))

ani = animate_pendulum(result_anim, lengths_anim, n=N_ANIM, title=f"{N_ANIM}-махало")

Колко махала да анимираме? (2-10):  5


In [13]:
try:
    MAX_N = int(input("Въведи максимален брой махала (препоръка: 8): "))
    MAX_N = max(2, MAX_N)
except ValueError:
    MAX_N = 8

if MAX_N > 8:
    print("Внимание: N > 8 може да е бавно!")

print()
lyapunov_results = {}

for n in range(2, MAX_N + 1):
    masses = [1.0] * n
    lengths = [1.0] * n
    angles = [np.pi/2] * n

    lyap = lyapunov_fast(n, masses, lengths, angles)
    lyapunov_results[n] = lyap
    print(f"N = {n}: λ = {lyap:.4f} ({'хаос' if lyap > 0 else 'ред'})")

Въведи максимален брой махала (препоръка: 8):  5



N = 2: λ = 0.8961 (хаос)
N = 3: λ = 0.8549 (хаос)
N = 4: λ = 0.9968 (хаос)
N = 5: λ = 0.9134 (хаос)


In [14]:
ns = list(lyapunov_results.keys())
lyaps = list(lyapunov_results.values())

plt.figure(figsize=(10, 6))
plt.plot(ns, lyaps, 'o-', color='royalblue', linewidth=2, markersize=8)
plt.axhline(y=0, color='tomato', linestyle='--', linewidth=1.5, label='граница хаос/ред')
plt.xlabel('Брой махала N')
plt.ylabel('Показател на Ляпунов λ')
plt.title('Как расте хаосът с броя на махалата')
plt.xticks(ns)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [15]:
def box_counting_dimension(points, min_box=0.01, max_box=1.0, n_scales=20):
   
    scales = np.logspace(np.log10(min_box), np.log10(max_box), n_scales)
    counts = []

    for scale in scales:
        
        boxes = set()
        for x, y in points:
            box_x = int(np.floor(x / scale))
            box_y = int(np.floor(y / scale))
            boxes.add((box_x, box_y))
        counts.append(len(boxes))

    
    log_scales = np.log(1.0 / scales)
    log_counts = np.log(counts)
    dimension = np.polyfit(log_scales, log_counts, 1)[0]

    return dimension, scales, counts

In [16]:
print("Изчисляване на фрактално измерение за N = 2 до MAX_N...")
fractal_results = {}

for n in range(2, MAX_N + 1):
    masses = [1.0] * n
    lengths = [1.0] * n
    angles_init = [np.pi/2] * n

    result = simulate_fast(n, masses, lengths, angles_init, t_span=(0, 60))

   
    angle_data = result.y[:n]
    last_x = np.zeros(len(result.t))
    last_y = np.zeros(len(result.t))

    for frame in range(len(result.t)):
        x, y = 0.0, 0.0
        for i in range(n):
            x = x + lengths[i] * np.sin(angle_data[i, frame])
            y = y - lengths[i] * np.cos(angle_data[i, frame])
        last_x[frame] = x
        last_y[frame] = y

    points = list(zip(last_x, last_y))
    dim, scales, counts = box_counting_dimension(points)
    fractal_results[n] = dim
    print(f"N = {n}: фрактално измерение = {dim:.4f}")

Изчисляване на фрактално измерение за N = 2 до MAX_N...
N = 2: фрактално измерение = 1.4232
N = 3: фрактално измерение = 1.2920
N = 4: фрактално измерение = 1.1870
N = 5: фрактално измерение = 1.1264


In [17]:
ns = list(fractal_results.keys())
dims = list(fractal_results.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


axes[0].plot(ns, dims, 's-', color='darkorchid', linewidth=2, markersize=8)
axes[0].axhline(y=1.0, color='gray', linestyle=':', linewidth=1, label='права линия (D=1)')
axes[0].axhline(y=2.0, color='tomato', linestyle=':', linewidth=1, label='запълнена площ (D=2)')
axes[0].set_xlabel('Брой махала N')
axes[0].set_ylabel('Фрактално измерение D')
axes[0].set_title('Фрактална сложност срещу брой махала')
axes[0].set_xticks(ns)
axes[0].grid(True, alpha=0.3)
axes[0].legend()


lyaps = [lyapunov_results[n] for n in ns]
ax2 = axes[1].twinx()
axes[1].plot(ns, lyaps, 'o-', color='royalblue', linewidth=2, markersize=8, label='Ляпунов λ')
ax2.plot(ns, dims, 's--', color='darkorchid', linewidth=2, markersize=8, label='Фрактал D')
axes[1].set_xlabel('Брой махала N')
axes[1].set_ylabel('Показател на Ляпунов λ', color='royalblue')
ax2.set_ylabel('Фрактално измерение D', color='darkorchid')
axes[1].set_title('Ляпунов и фрактал — сравнение')
axes[1].set_xticks(ns)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()